In [ ]:
import typing
import subprocess
import re
import socket
import threading
import ipaddress


def localIpAndNetmask()->typing.Tuple[str,str]:
    hostname = socket.gethostname()
    local_ip = socket.gethostbyname(hostname)
    # Use 'ipconfig' to get netmask
    result = subprocess.run(["ipconfig"], capture_output=True, text=True)
    lines = result.stdout.splitlines()
    netmask = None
    capture = False
    for line in lines:
        if local_ip in line:
            capture = True
        elif capture and "Subnet Mask" in line:
            netmask = line.split(":")[-1].strip()
            break
    if not netmask:
        raise RuntimeError("Could not determine subnet mask.")
    return local_ip, netmask

def ping(ipOrHost:str)->typing.Optional[typing.Tuple[str,str]]:
    """
    :return: (ip,name) or None
    NOTE: the act of pinging will add to arp list
    """
    try:
        # Use ping first (faster)
        result = subprocess.run(['ping','-a','-n', '1', '-i', '20', '-w', '20', ipOrHost],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            text=True, timeout=5)
        match = re.search(r'Pinging\s+(.*?)\s+\[([^\]]+)\]', result.stdout)
        if match:
            return (match.group(2),match.group(1))
        else:
            print(f"Could not extract IP from ping output:\n{result.stdout}")
    except subprocess.TimeoutExpired:
        print("Ping command timed out")
    return None

def pingSubnet(
    ip:typing.Optional[str]=None,
    netmask:typing.Optional[str]=None
    )->typing.Iterable[typing.Tuple[str,str]]:
    """
    Ping the entire subnet using many parallel threads
    (will fill in arp table if nothing else)
    
    :ip: if not specified, use our ip and netmask
    :netmask: if not specified use 255.255.255.0
    """
    ret=[]
    if ip is None:
        thisComputerIpMask=localIpAndNetmask()
        if ip is None:
            ip=thisComputerIpMask[0]
        if netmask is None:
            netmask=thisComputerIpMask[1]
    elif netmask is None:
        netmask='255.255.255.0'
    network = ipaddress.IPv4Network(f"{ip}/{netmask}", strict=False)
    print(f"Pinging subnet: {network} (excluding {ip})")
    threads:typing.List[threading.Thred]=[]
    for deviceIp in network.hosts():
        deviceIpString=str(deviceIp)
        if deviceIpString==ip: # skip ourselves
            continue
        def pingOne(ip_str:str):
            ret.append(ping(ip_str))
        t=threading.Thread(target=pingOne,args=(deviceIpString,))
        t.start()
        threads.append(t)
    for t in threads:
        t.join()
    return ret

def getArpTable()->typing.Iterable[str,str]:
    """
    Get the current arp table in the form (ip,mac)
    """
     # get the arp table
    result = subprocess.run(['arp','-a'],
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        text=True, timeout=5)
    # Match lines like:  192.168.1.1       00-11-22-33-44-55     dynamic
    for entry in re.finditer(r'(\d+\.\d+\.\d+\.\d+)\s+([-\w]+)\s+\w+',result.stdout):
        yield entry.group(1),entry.group(2)

def getMacAddress(ip:str)->typing.Optional[str]:
    """
    Get the mac address for a given ip
    """
    for deviceIp,mac in getArpTable():
        if deviceIp==ip:
            return mac
    # might not have been in the arp table,
    # so try to get it back and search again
    ping(ip)
    for deviceIp,mac in getArpTable():
        if deviceIp==ip:
            return mac
    return None

_knownNetworkDevices:typing.Dict[str,'NetworkDevice']={}
def getNetworkDevices(
    ip:typing.Optional[str]=None,
    netmask:typing.Optional[str]=None,
    rescan:bool=True
    )->typing.Generator['NetworkDevice',None,None]:
    """
    Get all network devices on the current subnet

    :ip: if not specified, use our ip and netmask
    :netmask: if not specified use 255.255.255.0
    """
    # first send what we know, in case that helps
    yield from _knownNetworkDevices.values()
    if rescan:
        try:
            # Ping will make sure arp knows about everyone
            ipToHost={}
            for k,v in pingSubnet(ip,netmask):
                ipToHost[k]=v
            # get the arp table
            result = subprocess.run(['arp','-a'],
                stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                text=True, timeout=5)
            # Match lines like:  192.168.1.1       00-11-22-33-44-55     dynamic
            for ip,mac in getArpTable():
                # if it's not in our subnet pings, we don't care about it
                hostname=ipToHost.get(ip)
                if hostname is not None:
                    dev=NetworkDevice(ip,mac,hostname)
                    _knownNetworkDevices[ip]=dev
                    yield dev
        except Exception as e:
            print(f"Error reading ARP table: {e}")
            return []

def resolveHostname(ip:str)->typing.Optional[str]:
    try:
        return socket.gethostbyaddr(ip)[0]
    except Exception as e:
        pass
    return None

def resolveNetbiosName(ip:str)->typing.Optional[str]:
    try:
        result = subprocess.run(['nbtstat', '-A', ip],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            text=True, timeout=5)
        # Look for the name with <00> and the "UNIQUE" flag
        match = re.search(r'^\s*([^\s]+)\s+<00>\s+UNIQUE', result.stdout, re.MULTILINE)
        if match:
            return match.group(1).strip()
    except Exception as e:
        pass
    return None

class NetworkDevice:
    """
    A single device on the network
    """
    def __init__(self,
        ip:str,
        mac:typing.Optional[str]=None,
        hostname:typing.Optional[str]=None,
        netbiosName:typing.Optional[str]=None,
        ):
        """ """
        self.ip=ip
        self._mac=mac
        self._hostname=hostname
        self._netbiosName=netbiosName

    @property
    def mac(self)->str:
        if self._mac is None:
            self._mac=getMacAddress(self.ip)
        if self._mac is None:
            return ''
        return self._mac

    @property
    def hostname(self)->str:
        if self._hostname is None:
            self._hostname=resolveHostname(self.ip)
        if self._hostname is None:
            return ''
        return self._hostname

    @property
    def netbiosName(self)->str:
        if self._netbiosName is None:
            self._netbiosName=resolveNetbiosName(self.ip)
        if self._netbiosName is None:
            return ''
        return self._netbiosName
    
    def __repr__(self):
        ret=[
            f'ip = {self.ip}',
            f'mac = {self.mac}']
        if self.hostname:
            ret.append(f'hostname = "{self.hostname}"')
        if self.netbiosName:
            ret.append(f'netbios name = "{self.netbiosName}"')
        return '\n'.join(ret)


In [ ]:
localIpAndNetmask()

('172.18.28.44', '255.255.248.0')

In [2]:

devices = getNetworkDevices()
print("Devices on local network:")
for dev in devices:
    if dev.ip.startswith('255.'):
        continue
    if dev.ip.startswith('224.'):
        continue
    print(f"{dev}\n")

Devices on local network:
Failed to resolve computer name for 172.18.28.44: Command '['nbtstat', '-A', '172.18.28.44']' timed out after 5 seconds
ip = 172.18.28.44
mac = ---
hostname = "keilander.osii.com"

Failed to resolve computer name for 172.18.24.1: Command '['nbtstat', '-A', '172.18.24.1']' timed out after 5 seconds
ip = 172.18.24.1
mac = 00-11-22-33-44-55
hostname = "1.qarestr.sub-172-18-24.myvzw.com"

Failed to resolve computer name for 172.18.31.255: Command '['nbtstat', '-A', '172.18.31.255']' timed out after 5 seconds
ip = 172.18.31.255
mac = ff-ff-ff-ff-ff-ff
hostname = "255.qarestr.sub-172-18-31.myvzw.com"

Failed to resolve computer name for 224.0.0.22: Command '['nbtstat', '-A', '224.0.0.22']' timed out after 5 seconds
ip = 224.0.0.22
mac = 01-00-5e-00-00-16
hostname = "igmp.mcast.net"

Failed to resolve computer name for 224.0.0.251: Command '['nbtstat', '-A', '224.0.0.251']' timed out after 5 seconds
ip = 224.0.0.251
mac = 01-00-5e-00-00-fb
hostname = "mdns.mcast.net"

In [3]:
!net view

System error 6118 has occurred.

The list of servers for this workgroup is not currently available



In [ ]:
import socket
socket.gethostbyname('jackpc7')

gaierror: [Errno 11001] getaddrinfo failed